In [1]:
import ROOT

OBJ: TStyle	ildStyle	ILD Style : 0 at: 0x8f769f0
OBJ: TStyle	ildStyle	ILD Style : 0 at: 0x8ffcf20


In [2]:
ROOT.EnableImplicitMT(6)

In [3]:
files = [
    "/eos/experiment/clicdp/data/user/l/lreichen/miniDST/sw_sl_analysis/l5_o1/rv02-02.sv02-02.mILD_l5_o1_v02.E250-SetA.I500105.P4f_sw_sl.eL.pL.n000.d_dstm_15065_0_mini-DST.edm4hep.root",
    "/eos/experiment/clicdp/data/user/l/lreichen/miniDST/sw_sl_analysis/l5_o1/rv02-02.sv02-02.mILD_l5_o1_v02.E250-SetA.I500108.P4f_sw_sl.eR.pL.n004.d_dstm_15102_36_mini-DST.edm4hep.root",
    "/eos/experiment/clicdp/data/user/l/lreichen/miniDST/sw_sl_analysis/l5_o1/rv02-02.sv02-02.mILD_l5_o1_v02.E250-SetA.I500106.P4f_sw_sl.eL.pR.n005.d_dstm_15177_25_mini-DST.edm4hep.root",
    "/eos/experiment/clicdp/data/user/l/lreichen/miniDST/sw_sl_analysis/l5_o1/rv02-02.sv02-02.mILD_l5_o1_v02.E250-SetA.I500108.P4f_sw_sl.eR.pL.n004.d_dstm_15102_32_mini-DST.edm4hep.root",
]
# df = ROOT.RDataFrame("events", files)
df = ROOT.RDataFrame("events", "/eos/experiment/clicdp/data/user/l/lreichen/miniDST/sw_sl_analysis/l5_o1/*.root")

OBJ: TStyle	ildStyle	ILD Style : 0 at: 0x13e54b20


Warning in <TClass::Init>: no dictionary for class edm4hep::Vector2i is available


In [4]:
# For each event I need to find the true electron and it should be after FSR
# From that one I also need to determine if it is connected to a reconstructed one
# And how many isolated electrons there are and if in the case of only one it matches
# I.e. I need flags: e_reconstructed, ...

In [5]:
df = df.Define("iso_lep_idx", "IsolatedElectrons_objIdx.index[0]")
df = df.Define("n_iso_e", "IsolatedElectrons_objIdx.size()")
df = df.Define("iso_lep_charge", "PandoraPFOs.charge[iso_lep_idx]")
df = df.Define("iso_lep_lvec", "ROOT::Math::PxPyPzEVector(PandoraPFOs.momentum.x[iso_lep_idx], PandoraPFOs.momentum.y[iso_lep_idx], PandoraPFOs.momentum.z[iso_lep_idx], PandoraPFOs.energy[iso_lep_idx])")

In [6]:
# just need to find the first stable electron!
df = df.Define("mc_e_mask", "MCParticlesSkimmed.generatorStatus == 1 && abs(MCParticlesSkimmed.PDG) == 11")
df = df.Define("mc_e_idx", "ArgMax(mc_e_mask)")
df = df.Define("mc_e_lvec", "ROOT::Math::PxPyPzMVector(MCParticlesSkimmed.momentum.x[mc_e_idx], MCParticlesSkimmed.momentum.y[mc_e_idx], MCParticlesSkimmed.momentum.z[mc_e_idx], MCParticlesSkimmed.mass[mc_e_idx])")
df = df.Define("mc_e_theta", "mc_e_lvec.Theta() > ROOT::Math::Pi()*0.5 ? ROOT::Math::Pi()*0.5 - mc_e_lvec.Theta() : mc_e_lvec.Theta()")
df = df.Define("mc_e_theta_mrad", "mc_e_theta * 1000")
df = df.Define("mc_e_theta_deg", "mc_e_theta / ROOT::Math::Pi() * 180.")

In [7]:
ROOT.gInterpreter.Declare("#include \"analyzers.h\"")
df = df.Define("mc_iso_e_mask", "auto r = RVec<int>(MCParticlesSkimmed.PDG.size(), 0); r[mc_e_idx] = 1; return r")
df = df.Define("pfo_iso_e_mask_cluster", "mcp_mask_to_pfo_mask(mc_iso_e_mask, PandoraPFOs, _RecoMCTruthLink_from, _RecoMCTruthLink_to, RecoMCTruthLink.weight)")
df = df.Define("pfo_iso_e_mask_track", "mcp_mask_to_pfo_mask(mc_iso_e_mask, PandoraPFOs, _RecoMCTruthLink_from, _RecoMCTruthLink_to, RecoMCTruthLink.weight, false)")

df = df.Define("n_iso_e_pfos_recoed_cluster", "Sum(pfo_iso_e_mask_cluster, 0)")
df = df.Define("n_iso_e_pfos_recoed_track", "Sum(pfo_iso_e_mask_track, 0)")

# df = df.Define("pfo_cluster_track_equal", "All(pfo_iso_e_mask_cluster && pfo_iso_e_mask_track)")

df = df.Define("n_iso_e_pfos_recoed_matching", "Sum(pfo_iso_e_mask_cluster && pfo_iso_e_mask_track, 0)")

In [8]:
df = df.Define("one_iso_e_but_wrong", "n_iso_e == 1 && !Any(iso_lep_idx == Nonzero(pfo_iso_e_mask_track))")
df = df.Define("one_iso_e_correct", "n_iso_e == 1 && n_iso_e_pfos_recoed_track && (iso_lep_idx == ArgMax(pfo_iso_e_mask_track))")

In [9]:
h = {}
h["n_iso_e_pfos_recoed_cluster"] = df.Histo1D(("", "", 10, 0., 10.), "n_iso_e_pfos_recoed_cluster")
h["n_iso_e_pfos_recoed_track"] = df.Histo1D(("", "", 10, 0., 10.), "n_iso_e_pfos_recoed_track")
h["n_iso_e_pfos_recoed_matching"] = df.Histo1D(("", "", 10, 0., 10.), "n_iso_e_pfos_recoed_matching")
h["one_iso_e_but_wrong"] = df.Histo1D(("", "", 2, 0., 2.), "one_iso_e_but_wrong")
h["one_iso_e_correct"] = df.Histo1D(("", "", 2, 0., 2.), "one_iso_e_correct")

h["mc_e_theta_mrad"] = df.Histo1D(("", "", 150, 50., 200.), "mc_e_theta_mrad")
h["mc_e_theta_deg"] = df.Histo1D(("", "", 30, 5., 8.), "mc_e_theta_deg")
h["mc_e_theta_deg_full"] = df.Histo1D(("", "", 176, 1., 89.), "mc_e_theta_deg")
h["mc_e_theta_deg_any"] = df.Histo1D(("", "", 40, 5., 13), "mc_e_theta_deg")
h["mc_e_theta_deg_full_any"] = df.Histo1D(("", "", 90, 0., 90.), "mc_e_theta_deg")

h["mc_e_theta_mrad_correct"] = df.Filter("one_iso_e_correct").Histo1D(("", "", 150, 50., 200.), "mc_e_theta_mrad")
h["mc_e_theta_deg_correct"] = df.Filter("one_iso_e_correct").Histo1D(("", "", 30, 5., 8.), "mc_e_theta_deg")
h["mc_e_theta_deg_full_correct"] = df.Filter("one_iso_e_correct").Histo1D(("", "", 176, 1., 89.), "mc_e_theta_deg")
h["mc_e_theta_deg_any_correct"] = df.Filter("n_iso_e_pfos_recoed_track").Histo1D(("", "", 40, 5., 13.), "mc_e_theta_deg")
h["mc_e_theta_deg_full_any_correct"] = df.Filter("n_iso_e_pfos_recoed_track").Histo1D(("", "", 90, 0., 90.), "mc_e_theta_deg")

In [10]:
c = {}
for k, his in h.items():
    c[k] = ROOT.TCanvas()
    his.SetTitle(f";{k}")
    his.Draw()
    c[k].Draw()

In [11]:
effs = {}
graphs = {}
for k, h_pass in h.items():
    if k.endswith("_correct"):
        base_name = k.removesuffix("_correct")
        try:
            h_total = h[base_name]
            eff = ROOT.TEfficiency(h_pass.GetPtr(), h_total.GetPtr())
            g = eff.CreateGraph()
            ce = ROOT.TCanvas()
            # eff.Draw("AP")
            g.Draw("AP")
            ce.Draw()
            c[f"{base_name}_eff"] = ce
            effs[base_name] = eff
            graphs[base_name] = g
        except KeyError:
            pass


In [19]:
c1 = ROOT.TCanvas()
g = graphs["mc_e_theta_deg_any"]
g.SetTitle(";#theta_{e^{#pm}} [deg]; Efficiency")
# g.Draw("ACP")
g.Draw("ALP")
# g.GetXaxis().SetLimits(5., 8.)
g.GetXaxis().SetLimits(5., 13.)
t = ROOT.TLatex()
t.DrawLatexNDC(0.25, 0.93935, "#font[62]{ILD} #font[52]{work in progress}")
c1.SetRightMargin(0.05)
c1.Draw()
c1.SaveAs("plots/acceptance/theta_zoom.pdf")

Info in <TCanvas::Print>: pdf file plots/acceptance/theta_zoom.pdf has been created


In [18]:

c2 = ROOT.TCanvas()
g = graphs["mc_e_theta_deg_full_any"]
g.SetTitle(";#theta_{e^{#pm}} [deg]; Efficiency")
g.Draw("ALP")
g.GetXaxis().SetLimits(0., 90.)
t = ROOT.TLatex()
t.DrawLatexNDC(0.25, 0.93935, "#font[62]{ILD} #font[52]{work in progress}")
c2.SetRightMargin(0.05)
c2.Draw()
c2.SaveAs("plots/acceptance/theta_full.pdf")

Info in <TCanvas::Print>: pdf file plots/acceptance/theta_full.pdf has been created
